In [1]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3
# %matplotlib inline
# %matplotlib qt6
# import mne
# mne.viz.set_browser_backend("qt")  # or "matplotlib"
# mne.set_config("MNE_BROWSER_BACKEND", "qt")  # or "matplotlib"
%gui qt

import numpy as np   # For the example
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple, Optional, Callable, Union, Any
import IPython

import numpy as np
import pandas as pd
import cv2

# Jupyter-lab enable printing for any line on its own (instead of just the last one in the cell)
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from cv2_enumerate_cameras import enumerate_cameras
from main import list_cameras

from src.camera_manager import CameraManager
from src.config_loader import ConfigLoader

Automatic pdb calling has been turned OFF


c:\Users\pho\repos\EmotivEpoc\ACTIVE_DEV\continuous_video_recorder\.venv\lib\site-packages\vidgear\gears\__init__.py:5: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


In [2]:
# Load minimal config for camera manager
config = ConfigLoader.load_config()
camera_manager = CameraManager(config)


# from continuous_video_recorder.main import list_cameras

1400: HD Pro Webcam C920
1401: USB Camera
700: USB Camera
701: HD Pro Webcam C920
702: Basler GenICam Source
703: Basler GenICam Source 2
704: Basler GenICam Source 3
705: Basler GenICam Source 4
706: OBS Virtual Camera


In [3]:
camera_manager.get_available_cameras()

[6]

In [4]:
from PyQt6.QtMultimedia import QMediaDevices

cameras = QMediaDevices.videoInputs()
for camera in cameras:
    print(f"ID: {camera.id()} | Name: {camera.description()}")

ID: b'\\\\?\\usb#vid_046d&pid_082d&mi_00#6&5e2b9d0&0&0000#{e5323777-f976-4f5b-9b55-b94699c46e44}\\global' | Name: HD Pro Webcam C920
ID: b'\\\\?\\usb#vid_058f&pid_5608&mi_00#a&b58fec7&1&0000#{e5323777-f976-4f5b-9b55-b94699c46e44}\\global' | Name: USB Camera


In [ ]:
config['webcam']


In [5]:

devices = config['webcam'].get('devices', [0])
device_labels = [f"camera_{i}" for i in devices]
device_camera_configs = [config['webcam'][f"camera_{i}"] for i in devices]

device_camera_names = [a_config['name'] for i, a_config in enumerate(device_camera_configs)]

found_cams_df: pd.DataFrame = camera_manager.try_find_cams(cam_names=device_camera_names)
target_open_cv_indicies = found_cams_df['open_cv_index'].to_numpy().astype(int)
target_open_cv_indicies

target_large_open_cv_indicies = found_cams_df['index'].astype(int).to_numpy()
target_large_open_cv_indicies

num_max_check: int = int(np.nanmax(target_open_cv_indicies) + 1)
num_max_check


# config['webcam']
# device_labels
# device_camera_configs

## OUTPUTS: target_open_cv_indicies, target_large_open_cv_indicies, num_max_check
target_open_cv_indicies


1400: HD Pro Webcam C920
1401: USB Camera
700: USB Camera
701: HD Pro Webcam C920
702: Basler GenICam Source
703: Basler GenICam Source 2
704: Basler GenICam Source 3
705: Basler GenICam Source 4
706: OBS Virtual Camera


array([0, 1])

array([1400, 1401])

2

array([0, 1])

In [6]:
# an_enumerated_large_idx: int = int(target_large_open_cv_indicies[0])
# an_enumerated_large_idx

an_enumerated_large_idx: int = int(target_large_open_cv_indicies[0])
an_enumerated_large_idx


matched_opencv_index = None
# Try enumerated index to see if it's the same device
enum_cap = cv2.VideoCapture(an_enumerated_large_idx, cv2.CAP_DSHOW)
if enum_cap.isOpened():
    enum_ret, _ = enum_cap.read()
    if enum_ret:
        # Both work - prefer standard index
        matched_opencv_index = an_enumerated_large_idx
        enum_cap.release()
        # break
else:
    print(f'failed to open matched_opencv_index: {matched_opencv_index}')
enum_cap.release()
matched_opencv_index

1400

failed to open matched_opencv_index: None


In [ ]:
cap = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)

In [ ]:
max_check: int = num_max_check
cameras = []
# Process each enumerated camera
for cam_info_row in found_cams_df.iterrows():
    # Try to find matching standard OpenCV index first
    cam_info = cam_info_row
    matched_opencv_index = None
    for i in range(max_check):
        try:
            # Try opening with standard index
            test_cap = cv2.VideoCapture(i, cv2.CAP_DSHOW)
            if test_cap.isOpened():
                ret, _ = test_cap.read()
                if ret:
                    # Check if this might be the same camera by comparing name/VID/PID
                    # We can't directly compare, but if enumerated index also opens, 
                    # we'll prefer the standard index
                    test_cap.release()
                    # Try enumerated index to see if it's the same device
                    enum_cap = cv2.VideoCapture(cam_info_row.index, cv2.CAP_DSHOW)
                    if enum_cap.isOpened():
                        enum_ret, _ = enum_cap.read()
                        if enum_ret:
                            # Both work - prefer standard index
                            matched_opencv_index = i
                            enum_cap.release()
                            break
                    enum_cap.release()
                else:
                    test_cap.release()
            else:
                test_cap.release()
        except Exception:
            continue
    ## for i in range(max_check)

    # Use matched OpenCV index or enumerated index
    if matched_opencv_index is not None:
        camera_index = matched_opencv_index
        cap = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)
    else:
        # Use enumerated index directly
        camera_index = cam_info_row.index
        try:
            cap = cv2.VideoCapture(camera_index, cv2.CAP_DSHOW)
        except Exception:
            continue
    
    if not cap.isOpened():
        continue
    
    # Verify camera works
    ret, _ = cap.read()
    if not ret:
        cap.release()
        continue
    
    # Build camera info
    camera_info = {
        "index": camera_index,
        "name": cam_info.name if hasattr(cam_info, 'name') and cam_info.name else f"Camera {camera_index}",
        "vid": f"{cam_info.vid:04X}" if hasattr(cam_info, 'vid') and cam_info.vid and cam_info.vid != 0 else None,
        "pid": f"{cam_info.pid:04X}" if hasattr(cam_info, 'pid') and cam_info.pid and cam_info.pid != 0 else None,
        "backend": getattr(cam_info, 'backend_name', None) or "DirectShow",
        "resolution": None,
    }
    
    # Get resolution
    try:
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        if width > 0 and height > 0:
            camera_info["resolution"] = (width, height)
    except Exception:
        pass
    
    cameras.append(camera_info)
    cap.release()
## END for cam_info in enumerated_cams...


cameras

In [ ]:
_out = camera_manager._list_cameras_cv2_enumerate(max_check=10)
_out

In [ ]:
found_cams_df = camera_manager.try_find_cams(cam_names = ['HD Pro Webcam C920', "USB Camera"])
found_cams_df


target_open_cv_indicies = found_cams_df['open_cv_index'].to_numpy()
# (open_cv_index)
target_open_cv_indicies


num_max_check: int = int(np.nanmax(target_open_cv_indicies) + 1)
num_max_check

# found_cams

In [ ]:
cameras = camera_manager.list_cameras_with_info(max_check=num_max_check)
cameras

In [ ]:
found_opencv_cams = list_cameras(max_check=num_max_check)
found_opencv_cams